# **Qwen3-8B&Lora**

In [1]:
# set config
CONFIG = dict(
    MODEL_PATH="Qwen/Qwen3-8B",
    LEARNING_RATE=2e-4,
    EPOCH=2,
    BATCH_SIZE=1,
    GRAD_ACCUM_STEPS=16,
    MAX_LENGTH=1024,
    SEED=42,

    LORA_R=64,
    LORA_ALPHA=32,
    LORA_DROPOUT=0.05,
    LORA_TARGET_MODULES=["q_proj", "k_proj", "v_proj", "o_proj"],
)

project_path = "/content/drive/MyDrive/Lectures/2025-26 Spring/CS445/Project"

In [2]:
!pip install -qU torchao

# import libs
import os
import json
import zipfile

import seaborn as sns
from pprint import pprint
import matplotlib.pyplot as plt

import re
import math
import torch
import numpy as np
import pandas as pd
from tqdm import tqdm
from scipy.stats import spearmanr
from torch.utils.data import Dataset, DataLoader
from transformers import get_linear_schedule_with_warmup
from transformers import AutoTokenizer, AutoModelForCausalLM, get_cosine_schedule_with_warmup
from peft import LoraConfig, get_peft_model, TaskType, set_peft_model_state_dict, load_peft_weights

# set seeds
def seed_all():
    seed = CONFIG["SEED"]
    os.environ["PYTHONHASHSEED"] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_all()

# enable optimization and determinism
torch.backends.cudnn.allow_tf32 = True
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

# set theme
sns.set_style("whitegrid")

# mount drive
from google.colab import drive
drive.mount('/content/drive')

# set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# print config
print("\nConfig >")
pprint(CONFIG)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Using device: cuda

Config >
{'BATCH_SIZE': 1,
 'EPOCH': 2,
 'GRAD_ACCUM_STEPS': 16,
 'LEARNING_RATE': 0.0002,
 'LORA_ALPHA': 32,
 'LORA_DROPOUT': 0.05,
 'LORA_R': 64,
 'LORA_TARGET_MODULES': ['q_proj', 'k_proj', 'v_proj', 'o_proj'],
 'MAX_LENGTH': 1024,
 'MODEL_PATH': 'Qwen/Qwen3-8B',
 'SEED': 42}


## **Dataset**

In [3]:
# extract dataset
with open(os.path.join(project_path, "train.json"), "r") as train_file: train_df = pd.read_json(train_file).T
with open(os.path.join(project_path, "dev.json"), "r") as validation_file: validation_df = pd.read_json(validation_file).T
with open(os.path.join(project_path, "test.json"), "r") as test_file: test_df = pd.read_json(test_file).T

print(f"{len(train_df)} train examples, {len(validation_df)} validation examples, {len(test_df)} test examples")

2280 train examples, 588 validation examples, 930 test examples


In [4]:
# display an example data
homonym = train_df.loc[0, "homonym"]
display(train_df[train_df["homonym"] == homonym])

,homonym,judged_meaning,precontext,sentence,ending,choices,average,stdev,nonsensical,sample_id,example_sentence
0,potential,the difference in electrical charge between tw...,The old machine hummed in the corner of the wo...,The potential couldn't be measured.,She collected a battery reader and looked on e...,"[4, 5, 2, 3, 1]",3.0,1.581139,"[False, False, False, False, False]",1843,The circuit has a high potential difference.
1,potential,the inherent capacity for coming into being,The old machine hummed in the corner of the wo...,The potential couldn't be measured.,She collected a battery reader and looked on e...,"[5, 3, 4, 4, 3]",3.8,0.83666,"[False, False, False, False, False]",1844,The project has great potential for success.
2,potential,the difference in electrical charge between tw...,The old machine hummed in the corner of the wo...,The potential couldn't be measured.,The machine could make such wonderful clothing...,"[2, 1, 4, 3, 1]",2.2,1.30384,"[False, False, False, False, False]",1845,The circuit has a high potential difference.
3,potential,the inherent capacity for coming into being,The old machine hummed in the corner of the wo...,The potential couldn't be measured.,The machine could make such wonderful clothing...,"[4, 5, 5, 3, 5]",4.4,0.894427,"[False, False, False, False, False]",1846,The project has great potential for success.
4,potential,the difference in electrical charge between tw...,The old machine hummed in the corner of the wo...,The potential couldn't be measured.,,"[1, 1, 4, 4, 3]",2.6,1.516575,"[False, False, False, False, False]",1847,The circuit has a high potential difference.
5,potential,the inherent capacity for coming into being,The old machine hummed in the corner of the wo...,The potential couldn't be measured.,,"[5, 4, 3, 4, 5]",4.2,0.83666,"[False, False, False, False, False]",1848,The project has great potential for success.


In [5]:
class AmbiStoryDataset(Dataset):
    system_prompt = """You will rate how plausible a candidate meaning of a homonym is, given a surrounding short narrative.

Use this 1-5 scale:
- 1: The meaning is incompatible with the context; a reader would never consider it here.
- 2: Theoretically possible but heavily disfavored - another meaning fits the context much better.
- 3: The narrative is genuinely ambiguous. This meaning and at least one other are about equally plausible. Many items rightly belong here - do not avoid 3.
- 4: This is the most natural reading, but other meanings remain conceivable.
- 5: This is the ONLY coherent reading; alternative meanings would make the narrative inconsistent or absurd.

Calibration notes:
- Use the full 1-5 range. Do not default to 4 or 5 just because the meaning fits.
- The homonym may appear in inflected form (e.g., past tense, plural) - count any morphological form.
- The ending often disambiguates - read it carefully before deciding.
- Reserve 5 for cases where the context actively rules out alternative meanings.

Return ONLY a single number from [1, 5]. Do not include explanation or any other text."""

    user_prompt = """Narrative: {precontext} **{sentence}** {ending}

Candidate: the word "{word}" means "{word_sense}" (example usage: "{example_sentence}")"""

    def __init__(self, data_df, is_training_split, tokenizer):
        super().__init__()
        self.tokenizer = tokenizer
        self.data_df = data_df.copy()
        self.is_training_split = is_training_split

        self.data_df[self.data_df[["average", "stdev"]] == "(???)"] = -1
        self.data_df["score"] = self.data_df["average"].astype(float)
        self.data_df["std"] = self.data_df["stdev"].astype(float)

        self.data_df["prompt"] = self.data_df.apply(self.generate_prompt, axis=1)

    def generate_prompt(self, row):
        ending = row["ending"] if isinstance(row["ending"], str) and row["ending"] else ""
        return self.user_prompt.format(
            precontext=row["precontext"],
            sentence=row["sentence"],
            ending=ending,
            word=row["homonym"],
            word_sense=row["judged_meaning"],
            example_sentence=row["example_sentence"]
        )

    def __len__(self):
        return len(self.data_df)

    def __getitem__(self, idx):
        messages = [
            {"role": "system", "content": self.system_prompt},
            {"role": "user", "content": self.data_df.loc[idx, "prompt"]}
        ]

        prompt = self.tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True
        )

        return {
            "prompt": prompt,
            "score": self.data_df.loc[idx, "score"],
            "std": self.data_df.loc[idx, "std"]
        }

In [6]:
class DataCollator:
    def __init__(self, tokenizer):
        self.tokenizer = tokenizer
        self.max_length = CONFIG["MAX_LENGTH"]

    def __call__(self, batch):
        prompts = [item["prompt"] for item in batch]
        scores = [item["score"] for item in batch]
        stds = [item["std"] for item in batch]

        full_texts = [p + str(t) + "<|im_end|>" for p, t in zip(prompts, scores)]

        tokenized = self.tokenizer(
            full_texts,
            truncation=True,
            max_length=self.max_length,
            padding=True,
            return_tensors="pt"
        )

        input_ids = tokenized["input_ids"]
        attention_mask = tokenized["attention_mask"]
        labels = input_ids.clone()
        labels[:] = -100

        prompt_only = self.tokenizer(
            prompts,
            truncation=True,
            max_length=self.max_length,
            padding=True,
            return_tensors="pt"
        )
        prompt_lens = prompt_only["attention_mask"].sum(dim=1).tolist()

        for i, plen in enumerate(prompt_lens):
            seq_len = int(attention_mask[i].sum().item())
            while plen < seq_len:
                labels[i, plen] = input_ids[i, plen]
                plen += 1

        inf_tokenized = self.tokenizer(
            prompts,
            truncation=True,
            max_length=self.max_length,
            padding=True,
            return_tensors="pt"
        )

        return {
            "input_ids": inf_tokenized["input_ids"],
            "attention_mask": inf_tokenized["attention_mask"],
            "train_input_ids": input_ids,
            "train_attention_mask": attention_mask,
            "labels": labels,
            "score": torch.FloatTensor(scores),
            "std": torch.FloatTensor(stds)
        }

In [7]:
# define dataset and dataloader
tokenizer = AutoTokenizer.from_pretrained(CONFIG["MODEL_PATH"])
tokenizer.padding_side = "left"
train_dataset = AmbiStoryDataset(train_df, True, tokenizer)
validation_dataset = AmbiStoryDataset(validation_df, False, tokenizer)
test_dataset = AmbiStoryDataset(test_df, False, tokenizer)

collate_fn = DataCollator(tokenizer)
train_dataloader = DataLoader(train_dataset, batch_size=CONFIG["BATCH_SIZE"], shuffle=True, collate_fn=collate_fn, pin_memory=True)
validation_dataloader = DataLoader(validation_dataset, batch_size=CONFIG["BATCH_SIZE"], shuffle=False, collate_fn=collate_fn, pin_memory=True)
test_dataloader = DataLoader(test_dataset, batch_size=CONFIG["BATCH_SIZE"], shuffle=False, collate_fn=collate_fn, pin_memory=True)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/728 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

In [8]:
# show example
row = train_dataset[0]
print(f"Prompt:\n{row['prompt']}")
print(f"Score: {row['score']}")
print(f"Std: {row['std']}")

Prompt:
<|im_start|>system
You will rate how plausible a candidate meaning of a homonym is, given a surrounding short narrative.

Use this 1-5 scale:
- 1: The meaning is incompatible with the context; a reader would never consider it here.
- 2: Theoretically possible but heavily disfavored - another meaning fits the context much better.
- 3: The narrative is genuinely ambiguous. This meaning and at least one other are about equally plausible. Many items rightly belong here - do not avoid 3.
- 4: This is the most natural reading, but other meanings remain conceivable.
- 5: This is the ONLY coherent reading; alternative meanings would make the narrative inconsistent or absurd.

Calibration notes:
- Use the full 1-5 range. Do not default to 4 or 5 just because the meaning fits.
- The homonym may appear in inflected form (e.g., past tense, plural) - count any morphological form.
- The ending often disambiguates - read it carefully before deciding.
- Reserve 5 for cases where the context ac

## **Training**

In [9]:
class LLM(torch.nn.Module):
    def __init__(self):
        super().__init__()

        self.model_path = CONFIG["MODEL_PATH"]
        lora_config = LoraConfig(
            r=CONFIG["LORA_R"],
            lora_alpha=CONFIG["LORA_ALPHA"],
            target_modules=CONFIG["LORA_TARGET_MODULES"],
            lora_dropout=CONFIG["LORA_DROPOUT"],
            task_type=TaskType.CAUSAL_LM,
        )

        self.model = AutoModelForCausalLM.from_pretrained(
            self.model_path,
            torch_dtype=torch.bfloat16,
            device_map="cuda"
        )
        self.model.config.use_cache = False
        self.model.gradient_checkpointing_enable()
        self.model.enable_input_require_grads()

        self.model = get_peft_model(self.model, lora_config)

    def forward(self, **kwargs):
        return self.model(**kwargs)

In [10]:
def train(model, dataloader, optimizer, scheduler):
    model.train()

    train_loss = 0
    progress_bar = tqdm(dataloader, desc="Training  ")
    grad_accum_steps = CONFIG["GRAD_ACCUM_STEPS"]

    optimizer.zero_grad()

    for batch_index, batch in enumerate(progress_bar):
        input_ids = batch["train_input_ids"].to(device, non_blocking=True)
        attention_mask = batch["train_attention_mask"].to(device, non_blocking=True)
        labels = batch["labels"].to(device, non_blocking=True)

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels
        )

        loss = outputs.loss/grad_accum_steps
        loss.backward()

        train_loss += loss.item()*grad_accum_steps*input_ids.size(0)

        if (batch_index + 1)%grad_accum_steps == 0 or batch_index == len(dataloader) - 1:
            grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad()

        progress_bar.set_postfix({"loss": f"{loss.item()*grad_accum_steps:.4f}"})

    train_loss /= len(dataloader.dataset)
    return train_loss

In [11]:
def extract_scores(texts, samples_per_prompt):
    scores = []
    for j in range(len(texts) // samples_per_prompt):
        samples = texts[j * samples_per_prompt : (j + 1) * samples_per_prompt]
        ints = []
        for t in samples:
            score_match = re.search(r"([1-4](\.\d+)?|5(\.0+)?)", t)
            if score_match:
                score = float(score_match.group(0))
                ints.append(score)
            else:
                ints.append(-1)
        scores.append(float(np.mean(ints)) if ints else -1.0)
    return scores

In [12]:
def test(model, dataloader, validation):
    model.eval()
    model.model.config.use_cache = True

    all_preds = []
    all_scores = []
    all_stds = []
    progress_bar = tqdm(dataloader, desc="Validation" if validation is True else "Testing   ")

    with torch.no_grad():
        for batch_index, batch in enumerate(progress_bar):
            input_ids = batch["input_ids"].to(device, non_blocking=True)
            attention_mask = batch["attention_mask"].to(device, non_blocking=True)
            target_scores = batch["score"].numpy()
            stds = batch["std"].numpy()

            generated_ids = model.model.generate(
                input_ids=input_ids,
                attention_mask=attention_mask,
                max_new_tokens=4,
                pad_token_id=tokenizer.pad_token_id,
                do_sample=False,
                use_cache=True,
                num_beams=5,
                num_return_sequences=5
            )

            new_tokens = generated_ids[:, input_ids.shape[1]:]
            decoded_outputs = tokenizer.batch_decode(new_tokens, skip_special_tokens=True)

            pred_scores = extract_scores(decoded_outputs, 5)

            all_preds.extend(pred_scores)
            all_scores.extend(target_scores)
            all_stds.extend(stds)

    model.model.config.use_cache = False

    if validation:
        all_preds = np.array(all_preds)
        all_scores = np.array(all_scores)
        all_stds = np.array(all_stds)

        valid_mask = all_preds != -1
        spearman_corr, _ = spearmanr(all_preds, all_scores)
        mae = np.mean(np.abs(all_preds - all_scores))
        rmse = np.sqrt(np.mean((all_preds - all_scores)**2))
        within_std = np.abs(all_preds - all_scores) <= np.maximum(all_stds, 1.0)
        acc_within_std = np.mean(within_std)*100
        scores_dict = {
            "spearman_corr": spearman_corr,
            "acc_within_std": acc_within_std,
            "mae": mae,
            "rmse": rmse
        }

        return scores_dict, all_preds

    else:
        zip_filename = "predictions.zip"
        jsonl_filename = "predictions.jsonl"
        with open(jsonl_filename, "w") as f:
            for pred_index, pred in enumerate(all_preds):
                final_pred = pred if pred != -1 else 3.0
                submission_dict = {"id": str(pred_index), "prediction": final_pred}
                f.write(json.dumps(submission_dict) + "\n")

        with zipfile.ZipFile(zip_filename, "w", zipfile.ZIP_DEFLATED) as zipf:
            zipf.write(jsonl_filename, arcname=jsonl_filename)

        return all_preds

In [13]:
def optimize(train_dataloader, validation_dataloader):
    seed_all()

    lr = CONFIG["LEARNING_RATE"]
    max_epochs = CONFIG["EPOCH"]
    train_weight_path = os.path.join("/content/train")
    train_history_path = os.path.join("/content/train/history.json")
    history = dict(train_loss=[], spearman=[], acc_std=[], mae=[], rmse=[], epoch=0, lr=lr, output="", config=CONFIG)

    print(f"Learning Rate is set to {lr}")
    history["output"] += f"Learning Rate is set to {lr}\n"

    print(f"\n{30*'-'} Training {30*'-'}")
    history["output"] += f"\n{30*'-'} Training {30*'-'}\n"

    model = LLM()
    optimizer = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=lr, weight_decay=0.01)

    num_update_steps_per_epoch = math.ceil(len(train_dataloader.dataset) / (CONFIG["BATCH_SIZE"] * CONFIG["GRAD_ACCUM_STEPS"]))
    total_steps = max_epochs * num_update_steps_per_epoch
    warmup_steps = int(0.06 * total_steps)

    scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=warmup_steps, num_training_steps=total_steps)

    best_spearman = float("-inf")
    best_epoch = 0
    for epoch in range(max_epochs):
        print(f"Epoch: {epoch+1}/{max_epochs} > ")
        history["output"] += f"Epoch: {epoch+1}/{max_epochs} > \n"

        train_loss = train(model, train_dataloader, optimizer, scheduler)
        scores, _ = test(model, validation_dataloader, True)

        val_spearman = scores["spearman_corr"]
        val_acc_std = scores["acc_within_std"]
        val_mae = scores["mae"]
        val_rmse = scores["rmse"]

        print(f"\tResults > Train Loss: {train_loss:.4f}, Spearman: {val_spearman:.4f}, Acc-STD: {val_acc_std:.2f}%, MAE: {val_mae:.4f}, RMSE: {val_rmse:.4f}\n")
        history["output"] += f"\tTrain Loss: {train_loss:.4f}, Spearman: {val_spearman:.4f}, Acc-STD: {val_acc_std:.2f}%, MAE: {val_mae:.4f}, RMSE: {val_rmse:.4f}\n\n"

        history["train_loss"].append(train_loss)
        history["spearman"].append(val_spearman)
        history["acc_std"].append(val_acc_std)
        history["mae"].append(val_mae)
        history["rmse"].append(val_rmse)

        if val_spearman > best_spearman:
            best_spearman = val_spearman
            best_epoch = epoch + 1
            history["epoch"] = best_epoch

            os.makedirs(os.path.dirname(train_weight_path), exist_ok=True)
            model.model.save_pretrained(train_weight_path)

        with open(train_history_path, "w") as history_file:
            json.dump(history, history_file)

    print(f"{30*'-'} Training {30*'-'}")
    history["output"] += f"{30*'-'} Training {30*'-'}\n"

    print(f"Best Epoch is {best_epoch}")
    history["output"] += f"\nBest Epoch is {best_epoch}\n"

    best_peft_weights = load_peft_weights(train_weight_path)
    set_peft_model_state_dict(model.model, best_peft_weights)
    model.to(device)

    return model, history

In [14]:
# train model
model, history = optimize(train_dataloader, validation_dataloader)

Learning Rate is set to 0.0002

------------------------------ Training ------------------------------


`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/399 [00:00<?, ?it/s]

...

Epoch: 1/2 > 


Validation: 100%|██████████| 588/588 [01:50<00:00,  5.34it/s]


	Results > Train Loss: 1.7155, Spearman: 0.7781, Acc-STD: 86.05%, MAE: 0.6235, RMSE: 0.8161

Epoch: 2/2 > 


Validation: 100%|██████████| 588/588 [01:49<00:00,  5.36it/s]


	Results > Train Loss: 0.6185, Spearman: 0.8012, Acc-STD: 86.90%, MAE: 0.5686, RMSE: 0.7311

------------------------------ Training ------------------------------
Best Epoch is 2


## **Results**

In [19]:
# test results
predictions = test(model, test_dataloader, False)

print()
print(f"Example Predictions > {predictions[:10]}")
print(f"Failed Predictions > {np.sum(predictions == -1)}")

"""
Results are as follows:

Spearman Correlation: 0.78
Spearman p-Value: 1.78e-191
Accuracy: 0.85 (790/930)
"""

Testing   : 100%|██████████| 930/930 [02:53<00:00,  5.37it/s]


Example Predictions > [4.6, 2.2, 4.04, 3.0, 4.200000000000001, 2.9999999999999996, 2.0, 3.6, 1.8, 2.44]
Failed Predictions > 0
